<!-- cabecera-entorno -->
## Antes de empezar

**Clase 14 · Introducción a Machine Learning** — Bloque 3 · Reto. Este cuaderno lo recorre **usted
solo**, leyendo: cada tarea trae la explicación y los comandos que necesita. El profesor circula por
el salón resolviendo dudas. Es el entregable de la clase.

**La rutina de siempre:** `git pull` antes de clase, y el entorno virtual activo (`(.venv)` en la
terminal). Si va a modificar este archivo, trabaje sobre una copia: duplique `reto.ipynb` como
`reto_mio.ipynb` y edite el duplicado. Así `git pull` nunca le reclama.

**Si la celda de abajo falla, no siga:** la respuesta está en el manual del entorno,
[`../INSTALACION.md`](../INSTALACION.md).

| Si ve esto | Qué pasó | Dónde se arregla |
|------------|----------|------------------|
| `ModuleNotFoundError` | El entorno virtual no está activo, o VSCode eligió otro intérprete | Manual, secciones 6.3 y 8.4, y problema 5 |
| `ImportError` al traer `load_diabetes` | scikit-learn quedó a medio instalar | Manual, problema 5 |
| El kernel no aparece en VSCode | Falta la extensión Jupyter o `ipykernel` dentro del entorno | Manual, problema 4 |

**Este reto no necesita ningún archivo de datos:** el dataset viene dentro de scikit-learn.

In [ ]:
# Verificación del entorno. Si algo falla aquí, la solución está en ../INSTALACION.md
import sys

try:
    import pandas as pd
    import numpy as np
    import matplotlib.pyplot as plt
    from sklearn.datasets import load_diabetes
except ModuleNotFoundError as error:
    raise ModuleNotFoundError(
        f"Falta la librería '{error.name}'. Active el entorno virtual y seleccione el intérprete "
        ".venv en VSCode (Ctrl+Shift+P > Python: Select Interpreter), luego reinicie el kernel. "
        "Ver ../INSTALACION.md, problema 5."
    ) from error

print("Intérprete:", sys.executable)
if ".venv" not in sys.executable:
    print("AVISO: este no parece el Python del entorno virtual. En VSCode: Ctrl+Shift+P >",
          "'Python: Select Interpreter' > el que dice .venv, y reinicie el kernel.")

print("Datos: load_diabetes devuelve", load_diabetes().data.shape, "-> no hace falta ningún CSV.")

# Clase 14 · Reto — Predecir la progresión de una enfermedad

**Dataset:** `load_diabetes` de scikit-learn (442 pacientes). **No hay CSV.**
**Consigna completa:** `README.md`

Un equipo médico tiene diez mediciones de rutina de 442 pacientes diabéticos —edad, sexo, índice de
masa corporal, presión arterial y seis análisis de sangre— y, para cada uno, un indicador de cómo
progresó la enfermedad un año después.

La pregunta: **¿se puede anticipar la progresión a partir de las mediciones de hoy?** Y detrás de
ella: ¿qué medición es la que más informa?

| Nombre | Qué es |
|--------|--------|
| `age` | Edad |
| `sex` | Sexo |
| `bmi` | Índice de masa corporal |
| `bp` | Presión arterial promedio |
| `s1` a `s6` | Seis mediciones de suero sanguíneo |
| `progresion` | La variable objetivo: progresión de la enfermedad a un año |

Las diez variables de entrada vienen centradas y escaladas, así que sus valores no se leen en unidades
originales. A un árbol eso le da igual: los árboles no necesitan variables escaladas.

**Otro dominio, otra escala, ningún archivo. La misma técnica del demo.**

## Cómo se recorre este cuaderno

Usted trabaja solo. Nadie va a dictar los pasos desde el tablero, así que cada tarea trae todo lo que
necesita para resolverse leyendo:

| Parte de la tarea | Qué contiene |
|-------------------|--------------|
| **La pregunta** | Lo que hay que responder, en español, como lo pediría el equipo médico |
| **El concepto** | Qué técnica aplica y por qué esa y no otra |
| **Los comandos** | Las instrucciones que va a usar, escritas de forma genérica |
| **Lo que decide usted** | Qué variable, qué conjunto y qué métrica. Ahí no hay respuesta escrita |
| **La celda de trabajo** | Los pasos numerados en comentarios. Usted escribe las líneas |
| **La comprobación** | `comprobar('TN', ...)` le dice si el resultado es correcto, sin mostrárselo |

**Por qué esto sigue siendo un reto y no una copia.** En el demo trabajó sobre consumo de agua de
Empocaldas, con 15.469 filas y variables categóricas. Aquí hay 442 pacientes, diez variables
numéricas y un dominio que no ha visto. Le damos el camino —los comandos, la técnica, el orden—, pero
el camino lo recorre usted sobre datos nuevos: elige el conjunto sobre el que evalúa, la métrica que
corresponde y **dice qué significa el número que sale**. La técnica se guía; el criterio no, y el
criterio es lo que se evalúa.

**La última tarea rompe el patrón:** le damos los comandos **sin el orden**. Armarlos es el ejercicio.

**Las celdas `Tu respuesta:`** no llevan código y se leen en la dimensión Saber. Un cuaderno con las
siete comprobaciones en verde y ninguna frase escrita está a medias.

### Aviso que cambia cómo se lee todo el reto

**En este dataset todos los modelos sobreajustan, incluso el más simple.** Con 442 filas y diez
variables ruidosas, un árbol individual no es la herramienta adecuada. Usted no hizo nada mal.

**Que un modelo salga malo no es un error suyo. No reportarlo sí lo es.**

### Recordatorios que ahorran errores

1. La métrica que cuenta es la de **prueba**, nunca la de entrenamiento.
2. `gap = entrenamiento - prueba`. Mayor a 0.10 es sobreajuste.
3. `random_state=SEMILLA` en todo lo que tenga azar, para que sus resultados sean reproducibles.
4. Regresión usa R2 y MAE. Clasificación usa accuracy. **No se mezclan.**

---

## Paso 0 · Cargar los datos

**El concepto.** `load_diabetes()` no lee ningún archivo: devuelve un objeto que scikit-learn trae
empaquetado, con tres cosas útiles.

| Atributo | Qué es |
|----------|--------|
| `.data` | Matriz de 442 x 10 con las variables de entrada |
| `.target` | Vector de 442 valores: la progresión de la enfermedad |
| `.feature_names` | Los nombres de las diez variables |

Esta celda ya está escrita. Ejecútela y confirme que dice 442 filas y 0 nulos.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.metrics import mean_absolute_error, r2_score, accuracy_score
from sklearn.dummy import DummyRegressor

plt.rcParams["figure.figsize"] = (9, 5)

SEMILLA = 42

datos = load_diabetes()

df = pd.DataFrame(datos.data, columns=datos.feature_names)
df["progresion"] = datos.target

print("Forma:", df.shape)
print("Nulos:", df.isna().sum().sum())
print()
print(df.head(3).round(4).to_string())
print()
print(df["progresion"].describe().round(1).to_string())

# X son las diez mediciones. y es lo que queremos predecir.
X = df[datos.feature_names]
y = df["progresion"]

In [ ]:
# Verificador de las siete tareas. Ejecute esta celda una vez y siga adelante.
# No hace falta entenderla hoy: es andamiaje del curso, no materia de la clase.
import hashlib

_RESULTADOS = {}

_CLAVES = ["T1", "T2", "T3", "T4", "T5", "T6", "T7"]

_PISTAS = {
    "T1": "len(X_train) y len(X_test), en ese orden, dentro de una lista. Si le dan 442 y 0, no ejecuto el split; si le dan otros numeros, revise test_size=0.2.",
    "T2": "Dentro del bucle, el R2 de prueba es r2_score(y_test, arbol.predict(X_test)), redondeado a 3 decimales, y se acumula con .append(). Si la lista tiene un solo elemento, el append quedo fuera del bucle. Si los valores son altos (0.9 y mas), los calculo sobre entrenamiento.",
    "T3": "gap = R2 de entrenamiento - R2 de prueba, en ese orden, redondeado a 3 decimales. Seis valores, en el mismo orden de PROFUNDIDADES. Todos van a dar mayores que 0.10: eso no es un error suyo.",
    "T4": "El eje es lo que devuelve fig, ax = plt.subplots(). Dibuje las dos curvas sobre ese mismo ax con label=..., rotule los dos ejes con ax.set_xlabel / ax.set_ylabel, ponga ax.legend() y un ax.set_title que diga el hallazgo. Guarde el eje en la variable pedida ANTES de plt.show().",
    "T5": "accuracy_score(yc_test, clf.predict(Xc_test)), redondeado a 3 decimales, uno por cada profundidad de la lista y en ese orden. Si uso r2_score, cambio de metrica: la clasificacion se evalua con accuracy.",
    "T6": "Entrene con max_depth=3 y random_state=SEMILLA sobre X_train, y_train. Despues pd.Series(arbol.feature_importances_, index=X.columns), ordenada de mayor a menor, .head(2).index.tolist(). Son los NOMBRES, no los valores.",
    "T7": "Los dos R2 son sobre PRUEBA y van redondeados a 3 decimales: primero el de su arbol de profundidad 3, despues el del DummyRegressor. El del tonto tiene que dar cerca de cero o negativo: asi esta definido el R2."
}

_ESPERADO = {
    "T1": "742eec8f3f",
    "T2": "3b9a542e22",
    "T3": "def46b02c9",
    "T5": "1dd6c1e599",
    "T6": "e6657735ca",
    "T7": "e242fda2c0"
}


def _firma(valor):
    """Reduce un resultado a un texto reproducible, sin importar como se calculo."""
    if isinstance(valor, (list, tuple)):
        return "lista|" + "|".join(_firma(v) for v in valor)
    if isinstance(valor, pd.DataFrame):
        partes = ["DataFrame", str(valor.shape), str([str(c) for c in valor.columns]),
                  str([str(i) for i in valor.index])]
        for columna in valor.columns:
            serie = valor[columna]
            if pd.api.types.is_bool_dtype(serie) or not pd.api.types.is_numeric_dtype(serie):
                partes.append(f"{columna}:{[str(v) for v in serie.tolist()]}")
            else:
                partes.append(f"{columna}:{round(float(serie.sum()), 4)}")
        return "|".join(partes)
    if isinstance(valor, pd.Series):
        return "|".join(["Series", str(len(valor)), str([str(i) for i in valor.index]),
                         str([str(v) for v in valor.tolist()])])
    if not isinstance(valor, str):
        try:
            return f"numero|{round(float(valor), 4)}"
        except (TypeError, ValueError):
            pass
    return f"otro|{valor!r}"


def _huella(valor):
    return hashlib.sha256(_firma(valor).encode("utf-8")).hexdigest()[:10]


def _redondear(valor, decimales):
    if decimales is None or valor is None:
        return valor
    if isinstance(valor, (list, tuple)):
        return [_redondear(v, decimales) for v in valor]
    if isinstance(valor, (pd.DataFrame, pd.Series)):
        return valor.round(decimales)
    try:
        return round(float(valor), decimales)
    except (TypeError, ValueError):
        return valor


def comprobar(clave, valor, decimales=None):
    """Dice si el resultado es el correcto, SIN revelar cual era.

    Compara una huella digital del resultado contra la esperada. Cuando no
    coincide, imprime una pista dirigida al error mas probable de ese punto,
    nunca la respuesta.
    """
    _RESULTADOS[clave] = False
    if valor is None:
        print(f"[{clave}] Sin resolver todavia: la variable sigue valiendo None.")
        print(f"[{clave}] Pista: {_PISTAS[clave]}")
        return
    valor = _redondear(valor, decimales)
    if isinstance(valor, pd.DataFrame):
        print(f"[{clave}] Usted produjo un DataFrame de {valor.shape[0]} filas "
              f"y {valor.shape[1]} columnas.")
    elif isinstance(valor, pd.Series):
        print(f"[{clave}] Usted produjo una Series de {len(valor)} elementos.")
    elif isinstance(valor, (list, tuple)):
        print(f"[{clave}] Usted produjo una lista de {len(valor)} elementos: {list(valor)}")
    else:
        print(f"[{clave}] Usted produjo: {valor!r}")
    if _huella(valor) == _ESPERADO.get(clave):
        _RESULTADOS[clave] = True
        print(f"[{clave}] CORRECTO.")
    else:
        print(f"[{clave}] Todavia no coincide.")
        print(f"[{clave}] Pista: {_PISTAS[clave]}")


def comprobar_grafico(clave, eje, con_ejes=True, con_leyenda=False):
    """Revisa que el grafico exista y cumpla los requisitos que se piden.

    No hay una unica respuesta correcta para un grafico: lo que se comprueba es
    que este dibujado, titulado y etiquetado. Que el titulo diga el hallazgo y no
    'Grafico 1' lo decide usted; eso no lo puede medir ningun chequeo.
    """
    _RESULTADOS[clave] = False
    if eje is None:
        print(f"[{clave}] Sin resolver todavia: la variable del eje sigue valiendo None.")
        print(f"[{clave}] Pista: {_PISTAS[clave]}")
        return
    if not hasattr(eje, "get_title"):
        print(f"[{clave}] Eso no es un eje de matplotlib, es un {type(eje).__name__}. "
              f"El eje es lo que devuelve plt.subplots(): fig, ax = plt.subplots().")
        print(f"[{clave}] Pista: {_PISTAS[clave]}")
        return
    faltas = []
    if not eje.get_title().strip():
        faltas.append("falta el titulo: ax.set_title('...')")
    if con_ejes:
        if not eje.get_xlabel().strip():
            faltas.append("falta la etiqueta del eje x: ax.set_xlabel('...')")
        if not eje.get_ylabel().strip():
            faltas.append("falta la etiqueta del eje y: ax.set_ylabel('...')")
    if len(eje.collections) + len(eje.lines) + len(eje.patches) == 0:
        faltas.append("el eje esta vacio: el grafico no se dibujo sobre este eje")
    if con_leyenda and eje.get_legend() is None:
        faltas.append("falta la leyenda: pase label='...' en cada curva y llame ax.legend()")
    if faltas:
        print(f"[{clave}] Todavia no esta completo:")
        for falta in faltas:
            print(f"[{clave}]   - {falta}")
        print(f"[{clave}] Pista: {_PISTAS[clave]}")
    else:
        _RESULTADOS[clave] = True
        print(f"[{clave}] CORRECTO: el grafico esta dibujado, titulado y etiquetado.")
        print(f"[{clave}] Lo que este chequeo NO puede juzgar: si el titulo dice el hallazgo. "
              f"Eso lo decide usted.")


def resumen_puntos_de_control():
    """Estado de los puntos de control del cuaderno."""
    print("Punto de control")
    print("-" * 46)
    for clave in _CLAVES:
        estado = "correcto" if _RESULTADOS.get(clave) else "pendiente"
        print(f"  {clave}: {estado}")
    logrados = sum(1 for c in _CLAVES if _RESULTADOS.get(c))
    print("-" * 46)
    print(f"{logrados} de {len(_CLAVES)} en verde.")
    if logrados < len(_CLAVES):
        print("Lo que sigue pendiente no se arregla escribiendo cualquier cosa hasta que pase:")
        print("vuelva a la pista, y si no sale, levante la mano.")


print("Verificador listo. Las tareas se comprueban con comprobar('T1', su_variable).")

---

## Parte 1 · Partir antes de entrenar

### Tarea 1 · El 20% que el modelo no va a ver

**La pregunta.** ¿Con cuántos pacientes va a aprender el modelo, y con cuántos lo vamos a examinar?

**El concepto.** Antes de entrenar nada se aparta una parte de los datos, y no se toca hasta el final.
Es la analogía del examen: entrenar es estudiar el taller, y evaluar sobre ese mismo taller es
autocalificarse. La única medida que significa algo es la que se toma sobre casos que el modelo no
vio. `train_test_split` parte `X` e `y` **juntos**, fila por fila, para que cada paciente siga
llevando su propia respuesta.

**Los comandos.**

```python
A_train, A_test, b_train, b_test = train_test_split(A, b, test_size=0.2, random_state=SEMILLA)
len(A_train)   # cuantas filas quedaron para entrenar
```

**Lo que decide usted.** Qué va en el lugar de `A` y de `b`, y en qué orden recoge los cuatro
resultados. Si los invierte, todo lo demás va a correr sin errores y va a estar mal.

Guarde en `tamano_particion` una lista con dos números: filas de entrenamiento y filas de prueba, en
ese orden.

In [ ]:
# TU CÓDIGO AQUÍ
# 1. Parta X e y en entrenamiento y prueba, 80/20, con random_state=SEMILLA.
#    Use exactamente los nombres X_train, X_test, y_train, y_test: el resto del
#    cuaderno cuenta con ellos.
# 2. Guarde en tamano_particion la lista [filas de entrenamiento, filas de prueba].
# 3. Imprima los dos numeros.

tamano_particion = None

In [ ]:
comprobar('T1', tamano_particion)

**Tu respuesta:** ¿Este problema es de regresión o de clasificación? ¿Y es aprendizaje supervisado o
no supervisado? Una frase para cada uno, diciendo **en qué se fijó** para decidirlo.

*Tu respuesta:*

---

## Parte 2 · La curva de sobreajuste

**Qué se practica aquí.** Lo central de la clase: distinguir un modelo que aprendió de uno que
memorizó, mirando dos números en vez de uno.

```
gap = R2 de entrenamiento - R2 de prueba

gap < 0.05        bien
gap 0.05 a 0.10   aceptable
gap > 0.10        SOBREAJUSTE
```

### Tarea 2 · El R2 de prueba de cada profundidad

**La pregunta.** ¿Qué tan bien predice el modelo sobre pacientes que no vio, según cuántas preguntas
seguidas le permitamos hacer al árbol?

**El concepto.** `max_depth` es cuántas preguntas encadenadas puede hacer el árbol. Recorrerlo en un
bucle y evaluar cada versión sobre el **mismo** conjunto de prueba es lo que permite comparar: si
cambiara el conjunto en cada vuelta, los números no serían comparables entre sí.

**Los comandos.**

```python
for parametro in LISTA:
    arbol = DecisionTreeRegressor(max_depth=parametro, random_state=SEMILLA)
    arbol.fit(CONJUNTO_DE_ENTRENAMIENTO, RESPUESTAS_DE_ENTRENAMIENTO)
    puntaje = r2_score(RESPUESTAS_REALES, arbol.predict(CONJUNTO_CORRESPONDIENTE))
    acumulador.append(round(puntaje, 3))
```

**Lo que decide usted.** Cuál de los dos conjuntos entra en `fit` y cuál en la evaluación. Confundirlos
no produce ningún error: produce números altos y falsos.

Guarde en `r2_prueba` la lista de los seis R2 **de prueba**, redondeados a 3 decimales, en el orden de
`PROFUNDIDADES`.

In [ ]:
PROFUNDIDADES = [2, 3, 5, 10, 20, None]

# TU CÓDIGO AQUÍ
# 1. Cree la lista vacia r2_prueba.
# 2. Recorra PROFUNDIDADES con un bucle for.
# 3. En cada vuelta: cree el arbol con esa profundidad y random_state=SEMILLA,
#    entrenelo con los datos de ENTRENAMIENTO.
# 4. Calcule el R2 sobre PRUEBA y agreguelo a r2_prueba redondeado a 3 decimales.
#    (El .append va DENTRO del bucle.)
# 5. Imprima la lista.

r2_prueba = None

In [ ]:
comprobar('T2', r2_prueba, decimales=3)

### Tarea 3 · El gap, que es lo que decide

**La pregunta.** ¿Cuánto se le cae el desempeño a cada modelo al pasar del taller al examen?

**El concepto.** El R2 de prueba solo no basta. Dos modelos pueden tener la misma prueba y estar en
situaciones opuestas: uno aprendió un patrón, el otro memorizó y tuvo suerte. La distancia entre
entrenamiento y prueba —el gap— es la que separa los dos casos, y por eso es la columna que se reporta
al lado de la métrica, siempre.

**Los comandos.**

```python
puntaje_taller = r2_score(RESPUESTAS_DE_ENTRENAMIENTO, arbol.predict(CONJUNTO_DE_ENTRENAMIENTO))
puntaje_examen = r2_score(RESPUESTAS_DE_PRUEBA, arbol.predict(CONJUNTO_DE_PRUEBA))
gap = puntaje_taller - puntaje_examen
```

**Lo que decide usted.** El orden de la resta (invertirlo cambia el signo y el diagnóstico), y si
además arma una tabla para leerla completa. Una `pd.DataFrame(lista_de_diccionarios)` se lee mucho
mejor que seis prints sueltos.

Guarde en `gaps` la lista de los seis gaps, redondeados a 3 decimales, en el orden de `PROFUNDIDADES`.

**Aviso:** los seis van a superar 0.10. No es un error suyo, es lo que da este dataset.

In [ ]:
# TU CÓDIGO AQUÍ
# 1. Cree la lista vacia gaps.
# 2. Recorra PROFUNDIDADES otra vez (o guarde los dos puntajes en el bucle anterior).
# 3. En cada vuelta calcule el R2 de entrenamiento y el de prueba.
# 4. gap = entrenamiento - prueba, redondeado a 3 decimales, y al append.
# 5. Imprima una tabla con profundidad, train, test y gap.
#    Pista: max_depth=None no es texto, asi que f"{profundidad:>6s}" revienta.
#    Envuelvalo: f"{str(profundidad):>6s}".

gaps = None

In [ ]:
comprobar('T3', gaps, decimales=3)

### Tarea 4 · El gráfico que muestra la separación

**La pregunta.** ¿En qué punto el modelo deja de aprender el patrón y empieza a memorizar? Muéstrelo
en un gráfico que se entienda sin que usted esté al lado explicándolo.

**El concepto.** Dos curvas sobre el mismo eje: entrenamiento y prueba contra la profundidad. Lo que
se ve es la separación entre las dos, que es el sobreajuste dibujado. Un gráfico de una sola curva no
diría nada: el hallazgo **es** la distancia.

**Los comandos.**

```python
fig, ax = plt.subplots(figsize=(9, 5))
posiciones = np.arange(len(LISTA))
ax.plot(posiciones, SERIE_1, "o-", label="TEXTO")
ax.plot(posiciones, SERIE_2, "s-", label="TEXTO")
ax.set_xticks(posiciones)
ax.set_xticklabels([str(v) for v in LISTA])
ax.set_xlabel("...")
ax.set_ylabel("...")
ax.set_title("...")
ax.legend()
plt.show()
```

**Lo que decide usted.** Qué dice el título. `"Gráfico 1"` y `"R2 vs max_depth"` describen los ejes,
que ya están rotulados; el título dice **el hallazgo**, como en la clase 9.

Necesita los R2 de entrenamiento además de los de prueba: si no los guardó, recalcúlelos. Guarde el
eje en la variable `eje_curva` **antes** de `plt.show()`.

In [ ]:
# TU CÓDIGO AQUÍ
# 1. Arme la lista de R2 de entrenamiento (si no la tiene todavia).
# 2. fig, eje_curva = plt.subplots(figsize=(9, 5))
# 3. Dibuje las dos curvas sobre eje_curva, cada una con su label.
# 4. Ponga las etiquetas del eje x con las profundidades.
# 5. Rotule los dos ejes, ponga la leyenda y un titulo que diga el hallazgo.

eje_curva = None

In [ ]:
comprobar_grafico('T4', eje_curva, con_leyenda=True)

**Tu respuesta:** Dos preguntas, y las dos se responden con la tabla y el gráfico delante.

1. Si tuviera que entregar **un** modelo, ¿qué `max_depth` elegiría? No vale "el que da mejor
   prueba": justifique con el gap y con lo que le tocaría explicarle al equipo médico.
2. El modelo con `max_depth=20` tiene R2 de entrenamiento de 1.000. ¿Es el mejor modelo? ¿Por qué?

*Tu respuesta:*

---

## Parte 3 · El mismo problema, como clasificación

**Qué cambia aquí.** Nada de los datos: la misma tabla, los mismos pacientes. Lo que cambia es la
**pregunta**. En vez de "¿qué número va a dar la progresión?", preguntamos "¿va a ser baja, media o
alta?".

Que un problema sea de regresión o de clasificación no es una propiedad del dataset: **depende de la
decisión que se va a tomar con la predicción.** Si la decisión es "a qué pacientes hago seguimiento
prioritario", tres grupos alcanzan, y predecir tres grupos es una pregunta más fácil que predecir un
número exacto.

`pd.qcut(serie, 3, labels=[...])` corta por **terciles**: los tres grupos quedan del mismo tamaño. No
es lo mismo que `pd.cut`, que corta por valores fijos y puede dejar un grupo con cuatro pacientes.

**El punto de referencia:** con tres clases balanceadas, adivinar al azar acierta el 33%. Una accuracy
que no supere eso con holgura no vale nada. Esa comparación es obligatoria: sin ella, un 0.53 suena
bien y no se sabe contra qué.

### Tarea 5 · El clasificador y su punto de referencia

**La pregunta.** ¿Acierta el clasificador la categoría de progresión de un paciente nuevo mejor que
tirando una moneda de tres caras?

**El concepto.** Mismo algoritmo, otra variante: `DecisionTreeClassifier` en vez de
`DecisionTreeRegressor`, porque ahora `y` es una categoría. Y otra métrica: **accuracy**, la
proporción de aciertos. R2 no significa nada aquí, igual que accuracy no significaba nada en la Parte
2. El corte en terciles y la partición estratificada ya vienen hechos.

**Los comandos.**

```python
modelo = DecisionTreeClassifier(max_depth=parametro, random_state=SEMILLA)
modelo.fit(CONJUNTO_DE_ENTRENAMIENTO, CATEGORIAS_DE_ENTRENAMIENTO)
acierto = accuracy_score(CATEGORIAS_REALES, modelo.predict(CONJUNTO_CORRESPONDIENTE))
```

**Lo que decide usted.** Qué métrica corresponde y sobre qué conjunto se reporta. Y si además imprime
la accuracy de entrenamiento para ver el gap, que aquí también existe.

Guarde en `accuracy_prueba` la lista de las cinco accuracy **de prueba**, redondeadas a 3 decimales,
en el orden de `PROFUNDIDADES_CLF`.

In [ ]:
# El corte en tres categorias y la particion ya vienen hechos.
df["categoria"] = pd.qcut(df["progresion"], 3, labels=["Baja", "Media", "Alta"])
print(df["categoria"].value_counts().to_string())
print()

y_cat = df["categoria"]
Xc_train, Xc_test, yc_train, yc_test = train_test_split(
    X, y_cat, test_size=0.2, random_state=SEMILLA, stratify=y_cat)

PROFUNDIDADES_CLF = [2, 3, 5, 10, None]

# TU CÓDIGO AQUÍ
# 1. Cree la lista vacia accuracy_prueba.
# 2. Recorra PROFUNDIDADES_CLF.
# 3. En cada vuelta: DecisionTreeClassifier con esa profundidad y random_state=SEMILLA,
#    entrenado con Xc_train, yc_train.
# 4. Calcule la accuracy sobre PRUEBA y agreguela redondeada a 3 decimales.
# 5. Imprima profundidad, accuracy de entrenamiento, accuracy de prueba y gap,
#    y debajo la referencia de azar: 0.333.

accuracy_prueba = None

In [ ]:
comprobar('T5', accuracy_prueba, decimales=3)

**Tu respuesta:** ¿El clasificador funciona mejor que el regresor, o solo lo parece porque la métrica
es distinta? Compare cada uno **contra el punto de referencia de su propia métrica**, no entre ellos.

*Tu respuesta:*

---

## Parte 4 · Qué variables manda el modelo

### Tarea 6 · Las dos que concentran la señal

**La pregunta.** De las diez mediciones que toma el equipo médico, ¿cuáles dos usa el modelo para casi
todo?

**El concepto.** `feature_importances_` da un número por columna: cuánto contribuyó esa variable a
reducir el error al partir los datos. Suman 1. Es la propiedad que hace del árbol el algoritmo de este
curso: se puede explicar. Se calcula sobre el modelo que uno realmente entregaría, no sobre el más
profundo.

**Los comandos.**

```python
modelo = DecisionTreeRegressor(max_depth=parametro, random_state=SEMILLA)
modelo.fit(CONJUNTO_DE_ENTRENAMIENTO, RESPUESTAS_DE_ENTRENAMIENTO)
importancias = pd.Series(modelo.feature_importances_, index=COLUMNAS)
importancias.sort_values(ascending=False)
importancias.sort_values().plot(kind="barh", ax=eje)
```

**Lo que decide usted.** Sobre qué modelo las calcula (use `max_depth=3`, el que se defendió en la
Parte 2) y qué dice el título del gráfico.

Guarde en `top_dos_variables` la lista con los **nombres** de las dos variables más importantes, de
mayor a menor.

In [ ]:
# TU CÓDIGO AQUÍ
# 1. Entrene un DecisionTreeRegressor con max_depth=3 y random_state=SEMILLA.
# 2. importancias = pd.Series(arbol.feature_importances_, index=X.columns)
# 3. Ordenelas de mayor a menor e imprimalas.
# 4. Grafiquelas con barh, con ejes rotulados y un titulo que diga el hallazgo.
# 5. Guarde en top_dos_variables los dos primeros NOMBRES: .head(2).index.tolist()

top_dos_variables = None

In [ ]:
comprobar('T6', top_dos_variables)

**Tu respuesta:** Escriba la interpretación completa. Tiene que incluir las tres cosas:

1. **Qué variables mandan**, con su peso aproximado.
2. **Qué se le puede decir al equipo médico** a partir de eso, en una frase que sirva para tomar una
   decisión.
3. **Qué NO se puede concluir.** Mencione explícitamente el asunto de la causalidad, y también qué
   pasa con las variables correlacionadas entre sí (`s1` a `s6` miden cosas relacionadas).

*Tu respuesta:*

---

## Parte 5 · Armarlo usted

**Qué cambia aquí.** Nada de técnica: esta tarea no usa un comando que no haya usado ya. Lo que cambia
es que **los comandos vienen sin el orden**. Armar la secuencia es el ejercicio, y es exactamente lo
que le va a tocar hacer en el proyecto final, donde nadie le va a numerar los pasos.

### Tarea 7 · ¿Le gana su modelo a no tener modelo?

**La pregunta.** Un modelo tonto que siempre responde el promedio de la progresión, sin mirar ninguna
medición, ya acierta algo por pura inercia. ¿Cuánto mejor que eso es su árbol?

**El concepto.** Toda métrica necesita un piso contra el cual leerse. En clasificación ese piso es el
azar (0.333 con tres clases); en regresión es predecir siempre la media, que es lo que hace
`DummyRegressor(strategy="mean")` y lo que define el 0 del R2. Si un modelo no le gana al tonto, no
hay modelo: hay un promedio caro.

**Los comandos, sin orden.** Estos son todos los que necesita. Algunos van dos veces, uno no va.

```python
r2_score(RESPUESTAS_REALES, PREDICCIONES)
DummyRegressor(strategy="mean")
.fit(CONJUNTO, RESPUESTAS)
.predict(CONJUNTO)
round(VALOR, 3)
DecisionTreeRegressor(max_depth=3, random_state=SEMILLA)
accuracy_score(RESPUESTAS_REALES, PREDICCIONES)
```

**Lo que decide usted.** El orden, sobre qué conjunto se entrena cada uno, sobre cuál se evalúan los
dos, y cuál de los siete comandos no pinta nada aquí.

Guarde en `comparacion_tonto` una lista con dos números redondeados a 3 decimales: el R2 de prueba de
su árbol de profundidad 3 y el R2 de prueba del modelo tonto, en ese orden.

In [ ]:
# TU CÓDIGO AQUÍ
# Sin pasos numerados: la secuencia la arma usted.
# Al final, comparacion_tonto = [R2 de prueba del arbol, R2 de prueba del tonto],
# los dos redondeados a 3 decimales. Imprima tambien el MAE de los dos.

comparacion_tonto = None

In [ ]:
comprobar('T7', comparacion_tonto, decimales=3)

**Tu respuesta:** Con todo lo anterior en la mano, escriba el párrafo que iría en un informe. Tiene
que responder las cuatro preguntas del reporte de modelo: qué predijo y a quién le importa, si es
regresión o clasificación y por qué, cómo evaluó (métrica de **prueba** y gap), y qué decisión soporta
la predicción. Si su conclusión es que este modelo no soporta ninguna decisión individual, dígalo: es
una conclusión legítima y es la honesta.

*Tu respuesta:*

---

## Punto de control

Ejecute la celda de abajo para ver cuántas de las siete tareas quedaron correctas.

In [ ]:
resumen_puntos_de_control()

---

## Opcional · Solo si terminó todo

No se hace en clase y no entra en la retroalimentación mínima. Se cierra en casa.

1. **`min_samples_leaf`**, la segunda palanca contra el sobreajuste. Obliga a que cada hoja tenga al
   menos N observaciones, así que el árbol no puede aislar casos sueltos. Pruebe 1, 5, 10, 20 y 50 con
   `max_depth=5` y mire qué le pasa al gap. Hay una sorpresa ahí.
2. **Dibujar el árbol** con `plot_tree(modelo, feature_names=list(X.columns), filled=True,
   fontsize=7)`. Es la ventaja de este algoritmo sobre casi todos los demás: se puede leer.

In [ ]:
# TU CÓDIGO AQUÍ (opcional)

---

## Antes de entregar

1. **Kernel → Restart and Run All.** Si algo revienta, arréglelo. Un cuaderno que no corre de arriba a
   abajo le pone techo a la dimensión Hacer.
2. Verifique que todas las celdas **Tu respuesta:** están escritas. Las siete comprobaciones en verde
   sin una sola frase escrita es medio entregable.
3. Revise esta lista, que es la misma del `README.md`:

- [ ] La tabla de la Parte 2 tiene las seis profundidades y la columna de gap.
- [ ] El gráfico de la curva tiene ejes rotulados, leyenda y un título que dice el hallazgo.
- [ ] La profundidad elegida está justificada con el gap, no con el máximo de la prueba.
- [ ] La Parte 3 compara la accuracy contra el 33% de referencia.
- [ ] La interpretación de la Parte 4 dice explícitamente que no es causalidad.
- [ ] Ninguna métrica de entrenamiento está presentada como resultado del modelo.
- [ ] El sobreajuste está reportado, no escondido.

4. Guarde como `clase14_reto_APELLIDO.ipynb` y súbalo al aula virtual.

## Para el Momento 3

La capa de machine learning de su proyecto final responde cuatro preguntas:

1. **¿Qué predijo?** Y por qué le importa a alguien.
2. **¿Es regresión o clasificación?** Y por qué esa y no la otra.
3. **¿Cómo evaluó?** Métrica sobre **prueba**, más el gap.
4. **¿Qué decisión soporta la predicción?** Si no soporta ninguna, el modelo sobra.

Y sigue en pie el requisito de la clase 13: mínimo **dos intervalos de confianza y una prueba de
hipótesis**.